# Candidate Recovery Metric — Daily Step Count

Steps are the most direct wearable proxy for "back to normal function", and the only candidate here
that can be measured both before and after surgery on the same instrument. The decision this feeds
is which single outcome the project commits to, and two numbers carry it: the population with
enough days on each side of the surgery to be read at all, and **how much of that population still
has an outcome once the outcome is actually computed.**

**Every reading here is within patient.** A per-day median over whoever wore the device is
cross-sectional: it moves when the set of wearers moves, and wearer sets move hardest in exactly the
peri-operative days a recovery metric needs most. So each day is read against that patient's own
normal, using the four baselines in `baselines.py` — a frozen pre-operative lookback, a trailing
flat window, a trailing exponential window, and a short window read against a long one. The
cross-sectional curve appears once, dotted, as the context the baselines exist to replace.

**Steps are the one wearable measure where a zero is ambiguous.** A patient who took no steps and a
patient whose device was in a drawer produce the same row, or no row at all, and no eligibility rule
can separate them. That is the informative-missingness problem this metric carries into any model
built on it, and it is stated in the scorecard rather than solved here.

**Governance.** The person-day extract is participant-level and stays in the kernel; every printed
number and every figure comes through `srp_common.reportable_days`, which drops any day fewer than
`SUPPRESS_THRESHOLD` patients contributed to. Unsuppressed figures stay git-ignored while
`*_suppressed.png` is committable, and the threshold must be confirmed against current All of Us
policy before anything is published.

## Shared Imports and Methods

`srp_common` resolves the pinned CDR, creates this notebook's git-ignored figure and cache
directories, and builds the cohort plus one index date per patient. The activity CTE, the shared
steps outcome and every baseline parameter are settled here, so nothing below can fail on setup.
There is no analysis in this section and nothing here answers a question.

#### Imports, the Activity CTE, and the Parameters Every Baseline Resolves To

`srp_common.steps_outcome_cte` is the one definition of the per-patient recovery ratio, shared with
`metric_comparison.ipynb`, so the number characterised in Part 5 and the number the cross-metric
comparison correlates cannot drift apart. **The grid reaches much further back than the span any
figure is drawn over**, because a trailing baseline on the first reported day needs its own history
behind it and the static window is anchored to the surgery rather than to that day.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Walk up from the kernel's cwd to the repo root, so this notebook runs from its task
# directory or from the root without the path being hard-coded in either.
_root = Path.cwd().resolve()
while not (_root / "srp_common.py").exists():
    if _root.parent == _root:
        raise RuntimeError(
            "Repo root not found: no srp_common.py at or above "
            f"{Path.cwd().resolve()}. Run this notebook from inside the repository."
        )
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import baselines as B
import srp_common
from srp_common import CATEGORICAL, INK, INK_MUTED, SERIES, SUPPRESS_THRESHOLD, SURFACE

cfg = srp_common.init("fitbit_steps")
CDR = cfg.CDR
FIGURES, CACHE_DIR = cfg.FIGURES, cfg.CACHE_DIR
cached_gbq = cfg.cached_gbq
reportable = srp_common.reportable_days

# Which of a patient's surgery-days becomes the index date is still open --
# index_surgery/surgery_day_distribution.ipynb is what settles it. "first" is used until then
# because it is the only rule no post-operative event can influence, and every result below is
# stamped with the rule so a later change is visible.
INDEX_RULE = "first"
WINDOWS = srp_common.DEFAULT_WINDOWS
BASELINE_STATISTIC = B.BASELINE_CENTRE   # "median": daily steps are heavily right-skewed
REPORT_FIRST, REPORT_LAST = -30, 30      # the span every baseline figure is drawn over
ZOOM = (-10, 10)                         # the admission itself, at readable width
GRID_FIRST = B.grid_first_day(REPORT_FIRST)

# One definition of the outcome for the whole notebook, shared with metric_comparison.ipynb: this
# CTE pair gives both the per-day `activity` rows the grid is built from and the per-patient
# `steps_outcome` ratio Part 5 characterises, so coverage cannot be reported for one number while
# the outcome is computed from another.
METRIC_CTE = srp_common.steps_outcome_cte(cfg, windows=WINDOWS)
CTES = cfg.cohort_ctes(INDEX_RULE, extra=METRIC_CTE)

# Fail here, not several minutes into a BigQuery run, if the CDR does not have what this notebook
# reads. cohort_ctes checks its own tables separately.
cfg.require_columns({
    "activity_summary": ["person_id", "date", "steps"],
})

print(f"index rule           : {INDEX_RULE}")
print(f"baseline statistic   : {BASELINE_STATISTIC}")
print("windows              : "
      + ", ".join(f"{k} [{lo}, {hi}]" for k, (lo, hi) in WINDOWS.items()))
print(f"reported span        : rel_day {REPORT_FIRST:+d} .. {REPORT_LAST:+d}")
print(f"grid spans           : rel_day {GRID_FIRST:+d} .. {REPORT_LAST:+d}"
      f"   ({REPORT_LAST - GRID_FIRST + 1} days)")
print()
print("the four baselines, at the values inherited from the sweeps")
print(f"  static      : {B.STATIC_LOOKBACK_DAYS}d window ending day "
      f"{-B.STATIC_DAYS_PRIOR:+d}, i.e. rel_day "
      f"{-(B.STATIC_DAYS_PRIOR + B.STATIC_LOOKBACK_DAYS - 1):+d} .. "
      f"{-B.STATIC_DAYS_PRIOR:+d}, needing {B.STATIC_MIN_ELIGIBLE_DAYS} eligible days")
print(f"  rolling     : {B.ROLLING_WINDOW_DAYS}d trailing window, gap "
      f"{B.ROLLING_GAP_DAYS}d, needing {B.ROLLING_MIN_DAYS} of its "
      f"{B.ROLLING_WINDOW_DAYS} days ({1 - B.ROLLING_MAX_MISSING:.0%} of them)")
print(f"  ewma        : halflife {B.EWMA_HALFLIFE_DAYS}d, capacity "
      f"{B.ewma_full_effective_days():.1f} effective days, needing "
      f"{B.EWMA_MIN_EFFECTIVE_DAYS:.1f} of them")
print(f"  comparative : {B.SHORT_SPAN_DAYS}d short vs {B.LONG_SPAN_DAYS}d long, "
      f"{B.SPAN_COUPLING}, in units of the long window's IQR")
print()
print(f"the ratio's own baseline : median steps over rel_day {WINDOWS['baseline'][0]:+d} .. "
      f"{WINDOWS['baseline'][1]:+d}, needing {srp_common.STEPS_MIN_WINDOW_DAYS} worn days")
print(f"small-cell threshold     : {SUPPRESS_THRESHOLD} patients per reported day")

## Part 1 — The Person-Day Grid

Everything below runs on one object: a grid with one row per `rel_day` and one column per patient,
holding that patient's step count on the days the eligibility rule admits and nothing anywhere else.
**A grid row is a calendar day, not a worn day.** That is the property the baselines depend on: a
trailing 30-day window has to span the 30 days it claims, and if rows were worn days then a patient
who wore the device on 30 scattered occasions would get a "30-day" window reaching back a year.

**No lower bound is placed on a day's step count, and that refusal is the load-bearing judgement in
this Part.** A floor would be the natural way to drop non-wear days — a device in a drawer records
nothing much — but a patient two days past major surgery genuinely takes very few steps, so a floor
would delete the largest true signal in the data and replace the dip with a hole. The cost of that
choice is that non-wear and immobility are not separable here, which is why the scorecard records
informative missingness rather than claiming it was handled.

This is the one Part that holds participant-level values in the kernel. That is allowed — the
analysis runs inside the Verily workspace and the rows never leave it — but it is the reason every
number printed from here on comes through `reportable()`.

#### Coverage, the Eligibility Rule, and What a Step Floor Would Cost

Coverage comes first because no amount of signal rescues a metric the cohort does not carry, and the
paired count is read off the grid rather than re-queried, so the population reported here is exactly
the one the baselines run over. **The two hypothetical floors are printed and not applied**: they
are there so a later reader can see what excluding low-step days would remove before deciding to
exclude them, rather than discovering the cost after the fact.

In [ ]:
# The eligibility rule. Only an upper bound is applied: over this many steps in a day is a device
# miscounting rather than a person walking, and no plausible patient is lost by refusing it.
MAX_STEPS = 60_000

# NOT applied -- printed so the cost of a low-step floor is visible to anyone who proposes one.
# A floor is the obvious way to drop non-wear days and the wrong way here, because the days it
# would drop hardest are the immediate post-operative days a recovery metric is entirely about.
HYPOTHETICAL_FLOORS = (100, 500)

# Paired-window minimum, the same 7 the other wearable notebooks use, so the three stay comparable
# in the scorecard.
PAIRED_MIN_DAYS = 7

coverage_sql = CTES + """
    SELECT
        (SELECT COUNT(*) FROM cohort)                       AS cohort_patients,
        (SELECT COUNT(*) FROM index_surgery)                AS patients_with_index_date,
        (SELECT COUNT(DISTINCT person_id) FROM activity)    AS patients_with_any_steps,
        (SELECT COUNT(*) FROM activity)                     AS step_days
"""

cov = cached_gbq(coverage_sql, "steps_coverage").iloc[0]
cohort_patients = int(cov["cohort_patients"])

extract_sql = CTES + f"""
    SELECT person_id, rel_day, steps
    FROM activity
    WHERE rel_day BETWEEN {GRID_FIRST} AND {REPORT_LAST}
      AND steps IS NOT NULL
"""

# Dry run before executing: this is the one query in the notebook returning person-day rows rather
# than an aggregate, so it is also the one that can scan far more than expected.
cfg.dry_run_bytes(extract_sql)
days = cached_gbq(extract_sql, "steps_person_days")

# NOTHING BELOW PRINTS A ROW. `days` and `grid` are participant-level and stay in the kernel;
# every number printed or plotted from here on is a count or a quantile over the whole cohort.
days["eligible"] = days["steps"].le(MAX_STEPS)
grid = B.eligible_value_grid(days, "steps", "eligible", GRID_FIRST, REPORT_LAST)

recorded = len(days)
eligible_days = int(grid.notna().to_numpy().sum())
print(f"cohort patients                            : {cohort_patients:>10,}")
print(f"  ... with an index date                   : "
      f"{int(cov['patients_with_index_date']):>10,}")
print(f"  ... with >=1 step-day anywhere           : "
      f"{int(cov['patients_with_any_steps']):>10,}"
      f"   ({int(cov['patients_with_any_steps']) / cohort_patients:6.1%})")
print(f"step-days inside the grid's span           : {recorded:>10,}")
print(f"  ... passing the eligibility rule         : {eligible_days:>10,}"
      f"   ({eligible_days / recorded:6.1%})")
print(f"grid                                       : "
      f"{grid.shape[0]:,} days x {grid.shape[1]:,} patients"
      f"   ({grid.notna().to_numpy().mean():.1%} filled)")

print()
print("days a low-step floor would remove, IF one were applied -- it is not")
for floor in HYPOTHETICAL_FLOORS:
    lost = int((days["steps"] < floor).sum())
    # Split by side of the surgery, because the whole objection to a floor is that it does not
    # fall evenly: it takes the post-operative days the metric exists to measure.
    post = days.loc[days["rel_day"].between(1, REPORT_LAST)]
    lost_post = int((post["steps"] < floor).sum())
    print(f"  < {floor:>5,} steps   {lost:>9,} of {recorded:,} days ({lost / recorded:5.1%})"
          f"   |   post-op days 1..{REPORT_LAST}: {lost_post:>8,} of {len(post):,} "
          f"({lost_post / len(post):5.1%})")

base_lo, base_hi = WINDOWS["baseline"]
post_lo, post_hi = WINDOWS["postop"]
baseline_days = grid.loc[base_lo:base_hi].notna().sum()
postop_days = grid.loc[post_lo:post_hi].notna().sum()
paired = int(((baseline_days >= PAIRED_MIN_DAYS) & (postop_days >= PAIRED_MIN_DAYS)).sum())

print()
print(f"patients with >= {PAIRED_MIN_DAYS} eligible days in the baseline window : "
      f"{int((baseline_days >= PAIRED_MIN_DAYS).sum()):>9,}")
print(f"patients with >= {PAIRED_MIN_DAYS} eligible days in the postop window   : "
      f"{int((postop_days >= PAIRED_MIN_DAYS).sum()):>9,}")
print(f"paired on both windows                                  : {paired:>9,}"
      f"   ({paired / cohort_patients:6.1%} of cohort)")

coverage = reportable(grid)
srp_common.plot_rel_day_curve(
    cfg, coverage.loc[coverage["rel_day"].between(GRID_FIRST, REPORT_LAST)], "n_patients",
    title="Patients contributing an eligible step-day, by day relative to surgery",
    ylabel="patients with an eligible step-day", suppress=True, show_windows=True,
    save_as="grid_coverage")

print()
print(f"{'day':>6s}{'patients':>12s}{'median steps':>16s}")
for day in [REPORT_FIRST, -14, -7, -1, 1, 7, 14, REPORT_LAST]:
    row = coverage.loc[coverage["rel_day"] == day]
    if row.empty:
        print(f"{day:>+6d}{'below the threshold, not reported':>28s}")
    else:
        print(f"{day:>+6d}{int(row['n_patients'].iloc[0]):>12,}"
              f"{row['p50'].iloc[0]:>16,.0f}")

## Part 2 — What Each Baseline Calls Normal

A baseline is a claim about where a patient's normal sits, so the first thing to look at is the
claims themselves rather than the departures from them. Drawn as levels, all four are in steps per
day and can share an axis, which is the only figure in this notebook where that is true — and the
observed cross-sectional curve is drawn beside them, dotted, as the thing they exist to replace
rather than as a result.

The four disagree structurally rather than noisily. The static baseline is one frozen number per
patient from before the surgery, so it is a flat line by construction and stays where the patient
was months earlier. The two trailing baselines follow the patient forward, so after surgery they are
partly built from post-operative days and drift down toward whatever the patient is now doing.
**A trailing baseline therefore under-reports the very thing a recovery metric is trying to measure,
and a frozen one cannot adapt to a patient whose normal genuinely changed.**

A fifth baseline is already in the repo and is not drawn here: the recovery ratio in Part 5 divides
by each patient's median over `DEFAULT_WINDOWS["baseline"]`, a 30-day window ending the day before
surgery. It is a static baseline with shorter, laxer parameters than `baselines.STATIC_*`, and the
cost table below is what says whether that laxity is buying coverage or buying contamination.

#### The Four Levels, and What Each One Costs

The static row of the movement table is not expected to be exactly zero, and that is the point. Each
patient's static level is frozen, so any movement in it is purely the set of contributing patients
changing underneath a fixed set of values — **the composition effect itself, measured** — which
bounds how much of the movement in the other three rows could be composition rather than anything
that happened to anybody. The coverage table beside it is the price: a stricter baseline is not a
better one unless what it buys is worth the cohort it costs.

In [ ]:
# Computed once and reused by Parts 3 and 4: the rolling medians over a grid this size are the
# expensive step in the notebook and nothing below them depends on anything else.
levels = B.baseline_levels(grid, statistic=BASELINE_STATISTIC)
deviations = B.deviations_from_each_baseline(grid, statistic=BASELINE_STATISTIC)
level_curves = {b: reportable(levels[b]) for b in B.BASELINE_NAMES}
departure_curves = {b: reportable(deviations[b]) for b in B.BASELINE_NAMES}

# THE OBSERVED CURVE IS CROSS-SECTIONAL -- it moves when the set of contributing patients moves,
# which is the problem the baselines exist to remove, so it is drawn as context, not as a result.
srp_common.plot_baseline_curves(
    cfg, {"observed day": coverage,
          **{f"{b} baseline": level_curves[b] for b in B.BASELINE_NAMES}},
    title="Where each baseline puts a patient's normal step count, "
          "against the day actually recorded",
    ylabel="steps per day", save_as="baseline_levels",
    span=(REPORT_FIRST, REPORT_LAST), colours=[INK_MUTED, *CATEGORICAL],
    note="median across contributing patients; levels only, not departures")

print("median level across contributing patients, by window (steps per day)")
print(f"{'window':<22s}{'observed':>12s}" + "".join(f"{b:>14s}" for b in B.BASELINE_NAMES))
for lo, hi, label in [(-30, -15, "baseline, first half"), (-14, -1, "the fortnight before"),
                      (1, 7, "first post-op week"), (8, 30, "rest of the month")]:
    print(f"{label:<22s}", end="")
    window = coverage.loc[coverage["rel_day"].between(lo, hi), "p50"]
    print(f"{window.median():>12,.0f}" if len(window) else f"{'--':>12s}", end="")
    for b in B.BASELINE_NAMES:
        frame = level_curves[b]
        window = frame.loc[frame["rel_day"].between(lo, hi), "p50"]
        print(f"{window.median():>14,.0f}" if len(window) else f"{'--':>14s}", end="")
    print()

print()
print("movement of the baseline itself, first post-op week vs the fortnight before")
for b in B.BASELINE_NAMES:
    frame = level_curves[b]
    before = frame.loc[frame["rel_day"].between(-14, -1), "p50"].median()
    after = frame.loc[frame["rel_day"].between(1, 7), "p50"].median()
    if pd.isna(before) or pd.isna(after):
        print(f"  {b:<12s} not reportable in one of the two windows")
        continue
    print(f"  {b:<12s} {before:>8,.0f} -> {after:>8,.0f}   ({after - before:+9,.0f} steps)")

cost = B.coverage_by_baseline(grid, deviations)

print()
print(f"{eligible_days:,} eligible person-days over {grid.shape[1]:,} patients")
print(f"{'baseline':<14s}{'person-days':>13s}{'of eligible':>13s}"
      f"{'patients':>11s}{'of eligible':>13s}")
for _, row in cost.iterrows():
    flag = ("   << below the small-cell threshold"
            if 0 < row["patients"] < SUPPRESS_THRESHOLD else "")
    print(f"{row['baseline']:<14s}{int(row['person_days']):>13,}"
          f"{row['share_of_person_days']:>12.1%}"
          f"{int(row['patients']):>11,}{row['share_of_patients']:>12.1%}{flag}")

# The floors each refusal comes from, printed next to the losses they caused, so a cost can be
# read against the rule rather than against a number nobody can locate.
print()
print("the floor doing the dropping, in each case")
print(f"  static      : {B.STATIC_MIN_ELIGIBLE_DAYS} eligible days inside rel_day "
      f"{-(B.STATIC_DAYS_PRIOR + B.STATIC_LOOKBACK_DAYS - 1):+d} .. {-B.STATIC_DAYS_PRIOR:+d}"
      f"   ({B.STATIC_MIN_ELIGIBLE_DAYS / B.STATIC_LOOKBACK_DAYS:.0%} of that window)")
print(f"  rolling     : {B.ROLLING_MIN_DAYS} eligible days in the trailing "
      f"{B.ROLLING_WINDOW_DAYS} days")
print(f"  ewma        : {B.EWMA_MIN_EFFECTIVE_DAYS:.1f} effective days out of a capacity of "
      f"{B.ewma_full_effective_days():.1f}")
print(f"  comparative : {B.days_required(B.SHORT_SPAN_DAYS, B.ROLLING_MAX_MISSING, floor=2)} "
      f"days in the short window AND "
      f"{B.days_required(B.LONG_SPAN_DAYS, B.ROLLING_MAX_MISSING, floor=2)} in the long one, "
      f"the long one also needing a non-zero IQR")
print(f"  the ratio   : {srp_common.STEPS_MIN_WINDOW_DAYS} worn days in EACH of rel_day "
      f"{WINDOWS['baseline'][0]:+d}..{WINDOWS['baseline'][1]:+d} and "
      f"{WINDOWS['postop'][0]:+d}..{WINDOWS['postop'][1]:+d}, and a non-zero baseline median")

# The cost is not constant across the span and the table cannot show that. A trailing baseline is
# hungriest right after surgery, when the device is least likely to have been worn, so the days a
# recovery metric most needs are the days its baseline is most likely to refuse.
srp_common.plot_baseline_coverage(
    cfg, grid, deviations, (REPORT_FIRST, REPORT_LAST),
    title="Patients each baseline can actually score, by day relative to surgery",
    save_as="baseline_coverage")

print()
print(f"thinnest reported day for each baseline, within rel_day {REPORT_FIRST:+d} .. "
      f"{REPORT_LAST:+d}")
for b in B.BASELINE_NAMES:
    scored = deviations[b].notna().sum(axis=1).loc[REPORT_FIRST:REPORT_LAST]
    below = int((scored < SUPPRESS_THRESHOLD).sum())
    print(f"  {b:<12s} {int(scored.min()):>7,} patients on day {int(scored.idxmin()):+d}"
          f"   ({below} of {len(scored)} days below the threshold)")

# Only read in this cell. Dropping them keeps a grid's worth of baselines out of the kernel for
# the rest of the notebook.
del levels

## Part 3 — Departure From Normal, and Whether It Comes Back

This is what the baselines are for: how far each day sat from that patient's own normal, and how
long it took to get back. Differencing against a per-patient level removes the composition effect a
cross-sectional curve cannot, because a patient can only enter a day's median by contributing a day
**and** having a baseline of their own to be read against.

**Three of the four departures share a unit and the fourth does not.** Static, rolling and
exponential are a day's steps minus a level, so they are in steps; comparative is a difference of
two window centres divided by a spread, so it is in units of that patient's own long-window IQR.
They are drawn on separate axes for that reason. The comparative curve also answers a slower
question — its numerator is the last fourteen days rather than the day itself, so a short admission
cannot appear on it at all and its response is smeared across the fortnight that follows.

The return question is asked on the departure curves rather than on a cross-sectional median,
because a cohort curve returning to its old level tells you nothing about whether the patients on it
are the same patients. A sustained rule is used so that one good day is not a recovery.

#### The Departure Curves, the Nadir, and the Day the Median Patient Is Back

The two constants are the knobs this question turns on and sit at the top of the cell rather than in
setup, because changing one should not mean reading two cells. **The recovery target is a fraction
of the patient's own baseline rather than an absolute step count**, so it means the same thing for a
patient walking 12,000 steps a day and one walking 2,000 — which is exactly what a cross-sectional
target cannot do.

In [ ]:
UNIT_BASELINES = ("static", "rolling", "ewma")   # the three in steps
RECOVERY_FRACTION = 0.95   # within 5% of normal counts as back; an exact return is noise-chasing
SUSTAIN_DAYS = 7           # days the median must hold the target before it counts as sustained

srp_common.plot_baseline_curves(
    cfg, {f"{b} baseline": departure_curves[b] for b in UNIT_BASELINES},
    title="How far each day's step count sat from that patient's own normal, "
          "by day relative to surgery",
    ylabel="steps - that patient's baseline", save_as="departures", zero_line=True,
    span=(REPORT_FIRST, REPORT_LAST),
    note="median across contributing patients; zero is the day that patient usually has")

srp_common.plot_baseline_curves(
    cfg, {f"{b} baseline": departure_curves[b] for b in UNIT_BASELINES},
    title="The same three departures through the ten days either side of surgery",
    ylabel="steps - that patient's baseline", save_as="departures_zoom", zero_line=True,
    span=ZOOM,
    note="median across contributing patients; zero is the day that patient usually has")

# Drawn alone because its unit is not the others'. THE Y AXIS HERE IS IQRs OF THAT PATIENT'S OWN
# LONG WINDOW, not steps.
srp_common.plot_baseline_curves(
    cfg, {"comparative index": departure_curves["comparative"]},
    title="The recent fortnight of steps against the established normal, "
          "in that patient's own IQRs",
    ylabel="(short centre - long centre) / long IQR",
    save_as="comparative_index", zero_line=True, span=(REPORT_FIRST, REPORT_LAST),
    note="median across contributing patients; unitless, not comparable to the figures above")

print("median departure from baseline, by window")
print(f"{'window':<24s}" + "".join(f"{b:>14s}" for b in B.BASELINE_NAMES))
for lo, hi, label in [(-30, -15, "baseline, first half"), (-14, -1, "the fortnight before"),
                      (1, 3, "first three days"), (4, 7, "rest of week one"),
                      (8, 14, "week two"), (15, 30, "weeks three and four")]:
    print(f"{label:<24s}", end="")
    for b in B.BASELINE_NAMES:
        frame = departure_curves[b]
        window = frame.loc[frame["rel_day"].between(lo, hi), "p50"]
        if not len(window):
            print(f"{'--':>14s}", end="")
            continue
        # The comparative column is in IQRs, not steps: rounding it to whole units the way the
        # other three are rounded would print every value as zero.
        spec = "+14.3f" if b == "comparative" else "+14,.0f"
        print(f"{window.median():>{spec}}", end="")
    print()
print("  (the first three columns are steps; comparative is long-window IQRs)")

# The nadir and the return, read off the WITHIN-PATIENT curve rather than a cohort median. The
# static baseline is the one used for this: it is the only one of the four that does not drift
# down toward the post-operative days, so it is the only one on which "back to normal" means back
# to where the patient was rather than back to where they now are.
static_departures = departure_curves["static"]
static_levels = level_curves["static"]
baseline_level = static_levels.loc[
    static_levels["rel_day"].between(-14, -1), "p50"].median()
target_gap = -(1 - RECOVERY_FRACTION) * baseline_level

print()
print(f"static baseline level, the fortnight before  : {baseline_level:>10,.0f} steps")
print(f"target: within {1 - RECOVERY_FRACTION:.0%} of it, i.e. a departure of "
      f"{target_gap:>+9,.0f} steps or better")

post = static_departures.loc[static_departures["rel_day"] >= 0].reset_index(drop=True)
nadir = post.loc[post["p50"].idxmin()]
print(f"post-operative nadir                         : {nadir['p50']:>+10,.0f} steps "
      f"on day {int(nadir['rel_day']):+d}"
      f"   ({int(nadir['n_patients']):,} patients)")

at_or_above = (post["p50"] >= target_gap).astype(int)
sustained = at_or_above.rolling(SUSTAIN_DAYS).sum() == SUSTAIN_DAYS
if sustained.any():
    # idxmax gives the row the window ENDS on; the run starts SUSTAIN_DAYS-1 rows earlier.
    # Indexing back by position rather than subtracting days keeps this correct where the curve
    # has gaps, which it will wherever a day fell below the suppression threshold.
    median_back_on_day = int(post.loc[sustained.idxmax() - SUSTAIN_DAYS + 1, "rel_day"])
    print(f"median departure first sustains the target   : day {median_back_on_day:+d}")
else:
    median_back_on_day = None
    print(f"median departure never sustains the target within day "
          f"{int(post['rel_day'].max())} -- on this baseline the median patient is still "
          "below normal at the end of the reported span")

print()
print(f"{'day':>5s}" + "".join(f"{b:>14s}{'n':>9s}" for b in UNIT_BASELINES))
for day in range(ZOOM[0], ZOOM[1] + 1):
    print(f"{day:>+5d}", end="")
    for b in UNIT_BASELINES:
        row = departure_curves[b].loc[departure_curves[b]["rel_day"] == day]
        if row.empty:
            print(f"{'--':>14s}{'--':>9s}", end="")
        else:
            print(f"{row['p50'].iloc[0]:>+14,.0f}{int(row['n_patients'].iloc[0]):>9,}", end="")
    print()

## Part 4 — Whether the Choice of Baseline Changes the Answer

Two baselines that rank person-days the same way are interchangeable for anything downstream that
only cares about which days were worse than usual, whatever their levels look like side by side.
Spearman is the right coefficient because the comparative index is on a different scale entirely, so
only the **ordering** of person-days is comparable across a pair. Each pair is computed on the
person-days both baselines score, so the counts differ from the coverage table in Part 2.

Rank correlation says nothing about **where zero is**, and zero is the whole claim — "this day was
worse than that patient's normal". Two baselines can rank a month identically and still put a third
of it on opposite sides of normal, because one sits lower than the other throughout, and a metric
counting bad days would then count different days. Both questions are asked on the post-operative
month as well as the whole grid, because a pair can agree across a year of ordinary days and
disagree exactly where the surgery is.

#### Rank Agreement, Side of Normal, and the Share Called Abnormal

The two thresholds are conventions and are named in the cell rather than left to the reader's
judgement. **A baseline calling most of the post-operative month abnormal is not detecting recovery,
it is mis-centred** — so the last table is the share of scored post-operative person-days each
baseline puts below normal, and a large value there is a warning rather than a finding.

In [ ]:
# Above INTERCHANGEABLE the pair ranks days the same and either will do; below DISTINCT the two
# are answering different questions and the choice between them IS the analysis.
INTERCHANGEABLE = 0.95
DISTINCT = 0.70
POSTOP = WINDOWS["postop"]

postop_only = {b: dev.loc[POSTOP[0]:POSTOP[1]] for b, dev in deviations.items()}
whole = B.baseline_agreement(deviations)
post_agreement = B.baseline_agreement(postop_only)

print("pairwise Spearman on the person-days both baselines score")
print(f"{'pair':<28s}{'whole grid':>13s}{'n':>14s}{'postop only':>14s}{'n':>14s}")
for (_, w), (_, p) in zip(whole.iterrows(), post_agreement.iterrows()):
    print(f"{w['pair']:<28s}{w['spearman']:>13.3f}{int(w['person_days']):>14,}"
          f"{p['spearman']:>14.3f}{int(p['person_days']):>14,}")

print()
print(f"reading, on rel_day {POSTOP[0]:+d} .. {POSTOP[1]:+d}")
for _, row in post_agreement.iterrows():
    rho = row["spearman"]
    if pd.isna(rho):
        verdict = "not enough shared person-days to say"
    elif rho >= INTERCHANGEABLE:
        verdict = "interchangeable for anything that only ranks days"
    elif rho >= DISTINCT:
        verdict = "related but not substitutable"
    else:
        verdict = "different questions; the choice between them IS the analysis"
    print(f"  {row['pair']:<28s} {rho:>6.3f}   {verdict}")

print()
print(f"share of shared person-days the pair puts on the SAME side of normal, rel_day "
      f"{POSTOP[0]:+d} .. {POSTOP[1]:+d}")
print(f"{'pair':<28s}{'person-days':>14s}{'same side':>12s}{'both below':>12s}")
for i, left in enumerate(B.BASELINE_NAMES):
    for right in B.BASELINE_NAMES[i + 1:]:
        a, b = B.flatten_pair(postop_only[left], postop_only[right])
        if len(a) < SUPPRESS_THRESHOLD:
            print(f"{left + ' vs ' + right:<28s}{len(a):>14,}"
                  f"{'--':>12s}{'--':>12s}   too few shared person-days")
            continue
        print(f"{left + ' vs ' + right:<28s}{int(len(a)):>14,}"
              f"{float(((a < 0) == (b < 0)).mean()):>12.1%}"
              f"{float(((a < 0) & (b < 0)).mean()):>12.1%}")

print()
print("share of scored post-operative person-days each baseline calls below normal")
for b in B.BASELINE_NAMES:
    values = B.flatten(postop_only[b])
    if len(values) < SUPPRESS_THRESHOLD:
        print(f"  {b:<12s} too few scored person-days to report")
        continue
    print(f"  {b:<12s} {float((values < 0).mean()):>6.1%}   of {len(values):,} person-days")

## Part 5 — The Outcome, Patient by Patient

Parts 1 to 4 ask whether a within-patient signal exists and which baseline reads it; this Part
builds the number a model would actually be asked to predict and asks what is left of the cohort
once it is built. **A median departure curve can return to zero while half the individual patients
never do**, so the distribution of the per-patient ratio, not any curve, is what the outcome is.

The ratio is the ratio `srp_common.steps_outcome_cte` defines and `metric_comparison.ipynb` reads,
so it is deliberately not rebuilt here. It is itself a within-patient measure — each patient's
post-operative median over their own pre-operative median — with a shorter and laxer baseline
window than `baselines.STATIC_*`, which is the trade Part 2's cost table prices.

Two constructions are put side by side, a fixed-horizon ratio and a time to recovery, because the
censoring in the second is what decides which of them this project can use. Every quantile, bucket
and cross-tabulation cell is computed inside BigQuery, so the per-patient values the outcome is
built from never enter the notebook.

#### The Ratio's Distribution, Its Censoring, and Whether the Two Constructions Agree

The two constants at the top are the knobs this question turns on: a horizon longer than the 30-day
screening window, because a time-to-recovery outcome must look further out or censor almost everyone
by construction, and a minimum number of worn days before a trailing week counts as a sustained
level. **The recovery target and the sustain rule are the same ones Part 3 applied to the median
curve**, so the two answers are about one definition of recovery rather than two.

In [ ]:
# A time-to-recovery outcome has to look further out than the 30-day screening window, or almost
# everyone is censored by construction. Declared here rather than in DEFAULT_WINDOWS so the
# screening numbers stay comparable across metrics.
RECOVERY_HORIZON = 180

# A trailing week containing only one worn day is not evidence of a sustained level, so a run has
# to carry at least this many days before it can count as a recovery.
MIN_DAYS_IN_RUN = 4

# The wear requirement the ratio itself is defined under, taken from the shared CTE rather than
# restated, so this print cannot describe a stricter rule than the SQL applied.
MIN_WINDOW_DAYS = srp_common.STEPS_MIN_WINDOW_DAYS

ratio_sql = CTES + f"""
    SELECT
        COUNT(*)                                                     AS patients_with_activity,
        COUNTIF(steps_recovery_ratio IS NOT NULL)                    AS n_eligible,
        APPROX_QUANTILES(steps_recovery_ratio, 100)[SAFE_OFFSET(10)] AS p10_ratio,
        APPROX_QUANTILES(steps_recovery_ratio, 100)[SAFE_OFFSET(25)] AS p25_ratio,
        APPROX_QUANTILES(steps_recovery_ratio, 100)[SAFE_OFFSET(50)] AS p50_ratio,
        APPROX_QUANTILES(steps_recovery_ratio, 100)[SAFE_OFFSET(75)] AS p75_ratio,
        APPROX_QUANTILES(steps_recovery_ratio, 100)[SAFE_OFFSET(90)] AS p90_ratio,
        COUNTIF(steps_recovery_ratio < 0.5)                          AS n_under_half,
        COUNTIF(steps_recovery_ratio >= {RECOVERY_FRACTION})         AS n_at_baseline
    FROM steps_outcome
"""

ratio = cached_gbq(ratio_sql, "steps_outcome_ratio").iloc[0]
n_eligible = int(ratio["n_eligible"])

print(f"patients with any step-day     : {int(ratio['patients_with_activity']):>9,}")
print(f"  ... with the ratio defined   : {n_eligible:>9,}"
      f"   ({n_eligible / cohort_patients:6.1%} of cohort)")
print(f"    (needs >= {MIN_WINDOW_DAYS} worn days in each of the baseline and postop windows)")
print()
for label, key in [("p10", "p10_ratio"), ("p25", "p25_ratio"), ("p50", "p50_ratio"),
                   ("p75", "p75_ratio"), ("p90", "p90_ratio")]:
    print(f"  {label} recovery ratio            : {ratio[key]:9.3f}")
print()
print(f"below half their baseline      : {int(ratio['n_under_half']):>9,}"
      f"   ({int(ratio['n_under_half']) / n_eligible:6.1%} of eligible)")
print(f"at or above {RECOVERY_FRACTION:.0%} of baseline    : {int(ratio['n_at_baseline']):>9,}"
      f"   ({int(ratio['n_at_baseline']) / n_eligible:6.1%} of eligible)")

# Bucketed in SQL rather than by binning a per-patient frame in pandas, which would mean pulling
# one row per participant into the notebook. The top bucket is open-ended: a patient can be
# several times their baseline, and a long thin tail of near-empty buckets is both unreadable and
# the part most likely to fall under the threshold.
hist_sql = CTES + """
    SELECT
        LEAST(FLOOR(steps_recovery_ratio * 5) / 5, 2.0) AS ratio_bucket,
        COUNT(*)                                        AS n_patients
    FROM steps_outcome
    WHERE steps_recovery_ratio IS NOT NULL
    GROUP BY ratio_bucket
    ORDER BY ratio_bucket
"""

hist = cached_gbq(hist_sql, "steps_outcome_histogram")
hist["bucket"] = [f"{v:.1f}+" if v >= 2.0 else f"{v:.1f}" for v in hist["ratio_bucket"]]

srp_common.plot_distribution(
    cfg, hist, "bucket",
    title="Steps recovery ratio, one value per patient",
    xlabel="postop median steps / baseline median steps",
    suppress=True, save_as="steps_outcome_ratio")

# The same target and sustain rule Part 3 applied to the median departure curve, applied here per
# patient, so the two answers are about one definition of recovery rather than two.
RECOVERY_CTE = f"""
    , daily AS (
        SELECT
            a.person_id,
            a.rel_day,
            a.steps,
            o.baseline_steps
        FROM activity AS a
        JOIN steps_outcome AS o ON o.person_id = a.person_id
        WHERE o.steps_recovery_ratio IS NOT NULL
          AND a.rel_day BETWEEN 1 AND {RECOVERY_HORIZON}
    ),
    smoothed AS (
        SELECT
            person_id,
            rel_day,
            baseline_steps,
            AVG(steps) OVER trailing_week AS steps_run,
            COUNT(*)   OVER trailing_week AS days_in_run
        FROM daily
        -- RANGE, not ROWS: the frame has to be the {SUSTAIN_DAYS} calendar days ending on this
        -- one. A row frame would span months for a patient who wore the device on
        -- {SUSTAIN_DAYS} scattered days, and would call that a sustained recovery.
        WINDOW trailing_week AS (
            PARTITION BY person_id ORDER BY rel_day
            RANGE BETWEEN {SUSTAIN_DAYS - 1} PRECEDING AND CURRENT ROW
        )
    ),
    recovered AS (
        SELECT
            person_id,
            MIN(rel_day) AS day_recovered
        FROM smoothed
        WHERE days_in_run >= {MIN_DAYS_IN_RUN}
          AND steps_run >= {RECOVERY_FRACTION} * baseline_steps
        GROUP BY person_id
    )"""

censoring_sql = CTES + RECOVERY_CTE + """
    SELECT
        COUNT(*)                                              AS n_recovered,
        APPROX_QUANTILES(day_recovered, 100)[SAFE_OFFSET(25)] AS p25_day,
        APPROX_QUANTILES(day_recovered, 100)[SAFE_OFFSET(50)] AS p50_day,
        APPROX_QUANTILES(day_recovered, 100)[SAFE_OFFSET(75)] AS p75_day
    FROM recovered
"""

censoring = cached_gbq(censoring_sql, "steps_time_to_recovery").iloc[0]
n_recovered = int(censoring["n_recovered"])
share_recovered = n_recovered / n_eligible
median_days_to_recovery = (int(censoring["p50_day"]) if n_recovered >= SUPPRESS_THRESHOLD
                           else None)

print()
print(f"eligible patients                     : {n_eligible:>9,}")
print(f"  ... reaching {RECOVERY_FRACTION:.0%} of baseline for "
      f"{SUSTAIN_DAYS} days : {n_recovered:>9,}   ({share_recovered:6.1%})")
print(f"  ... censored at day {RECOVERY_HORIZON:<3d}              : "
      f"{n_eligible - n_recovered:>9,}   ({1 - share_recovered:6.1%})")
print()
if n_recovered < SUPPRESS_THRESHOLD:
    print(f"Fewer than {SUPPRESS_THRESHOLD} patients recover within the horizon, so the day "
          "quantiles are not reportable.")
else:
    print(f"days to recovery among those who reach it : "
          f"p25={int(censoring['p25_day'])}  p50={int(censoring['p50_day'])}  "
          f"p75={int(censoring['p75_day'])}")

# Which construction survives this is the point: heavy censoring makes the fixed-horizon ratio the
# tractable outcome and days-to-recovery a survival problem, not a regression one.
if share_recovered < 0.5:
    print(f"\n--> most eligible patients never recover within {RECOVERY_HORIZON} days. A "
          "days-to-recovery outcome would be censored for the majority.")

# A tertile cross-tabulation rather than a correlation, because time to recovery is censored for
# whoever never reaches baseline and a correlation would have to drop exactly those patients --
# the ones the disagreement would be about.
agreement_sql = CTES + RECOVERY_CTE + f"""
    , scored AS (
        SELECT
            o.steps_recovery_ratio,
            -- A censored patient is the slowest recovery observed, not a missing value: they are
            -- known not to have recovered by the horizon, so they sort past everyone who did.
            COALESCE(r.day_recovered, {RECOVERY_HORIZON} + 1) AS day_recovered
        FROM steps_outcome AS o
        LEFT JOIN recovered AS r ON r.person_id = o.person_id
        WHERE o.steps_recovery_ratio IS NOT NULL
    ),
    tertiles AS (
        SELECT
            NTILE(3) OVER (ORDER BY steps_recovery_ratio)  AS ratio_tertile,
            NTILE(3) OVER (ORDER BY day_recovered DESC)    AS speed_tertile
        FROM scored
    )
    SELECT ratio_tertile, speed_tertile, COUNT(*) AS n_patients
    FROM tertiles
    GROUP BY ratio_tertile, speed_tertile
    ORDER BY ratio_tertile, speed_tertile
"""

agreement = cached_gbq(agreement_sql, "steps_outcome_agreement")
smallest = srp_common.min_cell(agreement, "n_patients")

print()
print(f"smallest cell: {smallest:,}")
if smallest < SUPPRESS_THRESHOLD:
    print(f"At least one cell is below the {SUPPRESS_THRESHOLD}-patient threshold, so this "
          "table may not be reported as-is. Widen the tertiles to halves, or aggregate.")
else:
    table = agreement.pivot(index="ratio_tertile", columns="speed_tertile",
                            values="n_patients")
    print()
    print("rows = recovery ratio tertile, columns = time-to-recovery tertile; 1 = worst in "
          "both, since time to recovery is ordered descending so that a slow or censored "
          "recovery sorts as a poor outcome")
    print(table.to_string())
    on_diagonal = sum(int(table.loc[i, i]) for i in table.index if i in table.columns)
    total = int(agreement["n_patients"].sum())
    print(f"\nagreeing on tertile: {on_diagonal:,} of {total:,} ({on_diagonal / total:.1%})")

## Part 6 — Verdict

The audit criteria are recorded as one row in the shared task cache, which is what
`metric_comparison.ipynb` reads to build the cross-metric scorecard. **Nothing written here is a
participant-level value**: the row is counts, fractions and hand-entered judgements.

#### The Scorecard Row

Writing the verdict as data rather than prose is what lets the cross-metric comparison be computed
instead of transcribed. The hand-entered fields are judgements and are recorded as such, so a later
reader can disagree with one specifically rather than with the whole row. **The analysable
population is still the population the ratio is defined for**, not the paired count, because a
patient can clear both windows and still have a baseline median of zero.

In [ ]:
scored = {b: int(cost.set_index("baseline").loc[b, "patients"]) for b in B.BASELINE_NAMES}

srp_common.write_scorecard(
    cfg,
    family="wearable",
    source_table="activity_summary",
    index_rule=INDEX_RULE,
    outcome_type="continuous",
    outcome_definition=(
        f"median daily steps over days {WINDOWS['postop'][0]}..{WINDOWS['postop'][1]} "
        f"divided by median daily steps over days {WINDOWS['baseline'][0]}.."
        f"{WINDOWS['baseline'][1]}, both windows requiring >= {MIN_WINDOW_DAYS} worn days; "
        "read alongside the four baselines in baselines.py, none of which is yet settled"
    ),
    cohort_patients=cohort_patients,
    patients_with_metric=int(cov["patients_with_any_steps"]),
    # The analysable population is the one the outcome is actually defined for, which is narrower
    # than the paired count: a patient can clear both windows and still have a baseline median of
    # zero, leaving the ratio undefined.
    patients_analysable=n_eligible,
    coverage_share=n_eligible / cohort_patients,
    horizon_days=int(WINDOWS["postop"][1]),
    paired_on_eligible_days=paired,
    baseline_choice="unsettled",
    patients_scored_by_baseline="; ".join(f"{b}={scored[b]:,}" for b in B.BASELINE_NAMES),
    median_departure_back_on_day=median_back_on_day,
    median_recovery_ratio=float(ratio["p50_ratio"]),
    recovery_horizon_days=RECOVERY_HORIZON,
    share_recovered_by_horizon=share_recovered,
    median_days_to_recovery=median_days_to_recovery,
    informative_missingness="yes",
    leakage_risk="low",
    notes=("Read within patient against baselines.py, not cross-sectionally. Non-wear is "
           "plausibly informative -- a patient recovering badly is less likely to wear the "
           "device -- and for steps a non-wear day and a genuinely immobile day are "
           "indistinguishable, so missingness must be treated as censoring rather than noise. "
           "Baseline parameters are inherited from a cancer-diagnosis sweep and have not been "
           "re-swept against this cohort."),
)